[*chatbot-with-memory*](https://github.com/SAP-samples/generative-ai-codejam/blob/main/exercises/10-chatbot-with-memory.ipynb)

In [1]:
from config import init_env
from config import variables
init_env.set_environment_variables()

## Chatbot with Memory

Let's add some history to the interaction and build a chatbot. Unlike many people think. LLMs are fixed in their state. They are trained until a certain cutoff date and do not know anything after that point unless you feed them current information. That is also why LLMs do not remember anything about you or the prompts you send to the model. If the model seems to remember you and what you said it is always because the application you are using (e.g. ChatPGT or the chat function in SAP AI Launchpad) is sending the chat history to the model to provide the conversation history to the model as context.


### Import required packages  

In [2]:
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template, TemplateValue
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.service import OrchestrationService
from gen_ai_hub.orchestration.models.azure_content_filter import AzureContentFilter
from gen_ai_hub.orchestration.exceptions import OrchestrationError

### Create the chatbot class

In [3]:
class ChatBot:
    def __init__(self, orchestration_service: OrchestrationService):
        self.service = orchestration_service
        ## Define the Orchestration Configuration
        self.config = OrchestrationConfig(
            # Define the prompt template
            template=Template(
                messages=[
                    SystemMessage("You are a helpful chatbot assistant."),
                    UserMessage("{{?user_query}}"),
                ],
            ),
            # Define the LLM 
            llm = LLM(
                name="gpt-4o",
                version="latest",
                parameters={
                    "max_tokens": 500,
                    "temperature": 1
                },
            )
        )
        self.history: List[Message] = []

    def chat(self, user_input):
        response = self.service.run(
            config=self.config,
            template_values=[
                TemplateValue(name="user_query", value=user_input),
            ],
            history=self.history,
        )

        message = response.orchestration_result.choices[0].message

        #print("1",self.history)
        self.history = response.module_results.templating
        #print("2",self.history)
        #print("m",message)
        self.history.append(message)
        #print("3",self.history)

        return message.content
    
    def reset(self):
        self.history = []

service = OrchestrationService(api_url=variables.AICORE_ORCHESTRATION_DEPLOYMENT_URL)
bot = ChatBot(orchestration_service=service)

### Test the bot

#### Create 2 separate instances for the ChatBot class

In [4]:
bot1 = ChatBot(orchestration_service=service)
bot2 = ChatBot(orchestration_service=service) 

#### Start the chat

In [5]:
print ("bot1:",bot1.chat("Hello, I am Samuel. Who are you?"))
print ("bot1:",bot1.chat("Which city is the capital of China?"))
print ("bot2:",bot2.chat("Hello, my name is Yufeng."))
print ("bot2:",bot2.chat("Which city is the capital of Germany?"))

bot1: Hello Samuel! I'm an AI assistant here to help you with any questions or tasks you might have. How can I assist you today?
bot1: The capital city of China is Beijing.
bot2: Hello, Yufeng! How can I assist you today?
bot2: The capital of Germany is Berlin.


##### Optional: check how the history list has been updated

In [6]:
print(bot1.history)

[Message(role=<Role.SYSTEM: 'system'>, content='You are a helpful chatbot assistant.', refusal=None, tool_calls=None), Message(role=<Role.USER: 'user'>, content='Hello, I am Samuel. Who are you?', refusal=None, tool_calls=None), Message(role=<Role.ASSISTANT: 'assistant'>, content="Hello Samuel! I'm an AI assistant here to help you with any questions or tasks you might have. How can I assist you today?", refusal=None, tool_calls=None), Message(role=<Role.SYSTEM: 'system'>, content='You are a helpful chatbot assistant.', refusal=None, tool_calls=None), Message(role=<Role.USER: 'user'>, content='Which city is the capital of China?', refusal=None, tool_calls=None), Message(role=<Role.ASSISTANT: 'assistant'>, content='The capital city of China is Beijing.', refusal=None, tool_calls=None)]


In [7]:
print (len(bot1.history))
print (bot1.history[0])
print (bot1.history[1])
print (bot1.history[2])
print (bot1.history[3])
print (bot1.history[4])
print (bot1.history[5])

6
Message(role=<Role.SYSTEM: 'system'>, content='You are a helpful chatbot assistant.', refusal=None, tool_calls=None)
Message(role=<Role.USER: 'user'>, content='Hello, I am Samuel. Who are you?', refusal=None, tool_calls=None)
Message(role=<Role.ASSISTANT: 'assistant'>, content="Hello Samuel! I'm an AI assistant here to help you with any questions or tasks you might have. How can I assist you today?", refusal=None, tool_calls=None)
Message(role=<Role.SYSTEM: 'system'>, content='You are a helpful chatbot assistant.', refusal=None, tool_calls=None)
Message(role=<Role.USER: 'user'>, content='Which city is the capital of China?', refusal=None, tool_calls=None)
Message(role=<Role.ASSISTANT: 'assistant'>, content='The capital city of China is Beijing.', refusal=None, tool_calls=None)


#### Continue the chat

In [8]:
print ("bot1:",bot1.chat("Where is this city?"))
print ("bot2:",bot2.chat("Where is this city?"))
 

bot1: Beijing is located in the northern part of China. It is situated in the northeastern section of the country and is surrounded by Hebei Province, except for the southeast, where it borders Tianjin Municipality. Its geographic coordinates are approximately 39.9° N latitude and 116.4° E longitude. Beijing serves as the political, cultural, and educational center of China.
bot2: Berlin is located in northeastern Germany. It is situated on the banks of the rivers Spree and Havel. As the capital city, it is the largest city in the country and serves as a major cultural, political, and economic center.


##### Optional: check how the history list has been updated

In [9]:
print (len(bot1.history))
print (bot1.history[0])
print (bot1.history[1])
print (bot1.history[2])
print (bot1.history[3])
print (bot1.history[4])
print (bot1.history[5])
print (bot1.history[6])
print (bot1.history[7])
print (bot1.history[8])
 

9
Message(role=<Role.SYSTEM: 'system'>, content='You are a helpful chatbot assistant.', refusal=None, tool_calls=None)
Message(role=<Role.USER: 'user'>, content='Hello, I am Samuel. Who are you?', refusal=None, tool_calls=None)
Message(role=<Role.ASSISTANT: 'assistant'>, content="Hello Samuel! I'm an AI assistant here to help you with any questions or tasks you might have. How can I assist you today?", refusal=None, tool_calls=None)
Message(role=<Role.SYSTEM: 'system'>, content='You are a helpful chatbot assistant.', refusal=None, tool_calls=None)
Message(role=<Role.USER: 'user'>, content='Which city is the capital of China?', refusal=None, tool_calls=None)
Message(role=<Role.ASSISTANT: 'assistant'>, content='The capital city of China is Beijing.', refusal=None, tool_calls=None)
Message(role=<Role.SYSTEM: 'system'>, content='You are a helpful chatbot assistant.', refusal=None, tool_calls=None)
Message(role=<Role.USER: 'user'>, content='Where is this city?', refusal=None, tool_calls=Non

#### Check the memory

In [16]:
 
print ("bot1:",bot1.chat("What is my name?"))
print ("bot2:",bot2.chat("What is my name?"))

bot1: Your name is Samuel.
bot2: Your name is Yufeng.
